# Lung Segmentation with MONAI

**Problem Type:** Medical image segmentation  
**Primary Dataset:** Montgomery County CXR Set with left/right lung masks  
**External Generalization Set:** Shenzhen Hospital CXR Set (qualitative inference only)  
**Stack:** MONAI + PyTorch  
**Goal:** Train a binary lung-field segmenter and explain the medical preprocessing choices that make the pipeline reasonable for chest radiographs.

## Project Overview

This notebook builds a MONAI-based lung segmentation workflow around the public **Montgomery County chest X-ray dataset**, which includes paired left and right lung masks. The **Shenzhen chest X-ray dataset** is also downloaded, but only for **external qualitative inference**, because the official NLM Shenzhen release provides lesion annotations rather than ready-to-train lung masks.

Why this method is correct:
- Lung segmentation is a dense pixel-labeling task, so a 2D medical segmentation model is the right tool.
- MONAI is a strong fit because it offers medical-image-first transforms, losses, metrics, and training patterns.
- Chest X-rays are grayscale and anatomy-constrained, so preprocessing should be conservative rather than aggressively image-augmented.

Deliverables from this notebook:
- Real dataset download inside the notebook
- Montgomery train/validation/test split
- MONAI UNet training
- Dice and IoU evaluation on held-out Montgomery images
- Qualitative mask review on Montgomery and Shenzhen images

## Dataset Source, License Notes, and Final Dataset Choice

**Official source page:** https://www.lhncbc.nlm.nih.gov/LHC-publications/pubs/TuberculosisChestXrayDatasets.html

**Montgomery County CXR Set:**
- 138 posteroanterior chest X-rays
- Paired **left** and **right** lung masks in PNG format
- Best fit for supervised lung segmentation

**Shenzhen Hospital CXR Set:**
- 662 chest X-rays
- Public image release plus radiologist abnormality annotations
- Useful here as an **external-domain inference set**, not as a supervised lung-mask training set

**Final choice:**
- Train, validate, and test on **Montgomery** because it contains the required lung masks.
- Run external qualitative predictions on **Shenzhen** to inspect domain shift and failure modes.

This keeps the notebook honest: Dice and IoU are reported only where real lung masks exist.

In [1]:
import importlib
import subprocess
import sys


def ensure_package(import_name: str, install_name: str | None = None) -> None:
    install_target = install_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        print(f"Installing {install_target}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", install_target])


ensure_package("monai")
ensure_package("requests")
ensure_package("bs4", "beautifulsoup4")
ensure_package("PIL", "pillow")
ensure_package("sklearn", "scikit-learn")

print("Environment setup complete.")

e:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(
e:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment setup complete.


## Imports, Configuration, and Medical Preprocessing Choices

Medical preprocessing choices used in this notebook:
- **Single-channel grayscale input:** chest radiographs are natively grayscale, so we keep one channel instead of fabricating RGB.
- **Percentile intensity clipping and rescaling:** chest X-rays often have scanner-dependent exposure variation; clipping to robust percentiles reduces extreme intensity outliers.
- **Resize with nearest-neighbor masks:** images can use smooth interpolation, but masks must preserve hard boundaries.
- **Mild spatial augmentation only:** small translations and scale jitter are acceptable; vertical flips and strong elastic warps are avoided because they can violate anatomy.
- **No aggressive contrast stylization:** for medical images, realism matters more than visually dramatic augmentation.

In [2]:
import json
import math
import os
import random
import re
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

from monai.data import CacheDataset
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.networks.nets import UNet
from monai.transforms import (
    Compose,
    EnsureChannelFirstd,
    EnsureTyped,
    Lambdad,
    LoadImaged,
    RandAffined,
    Resized,
    ScaleIntensityRangePercentilesd,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 2
LEARNING_RATE = 1e-3
SHENZHEN_EXTERNAL_LIMIT = 8

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoint"
for directory in (DATA_DIR, ARTIFACT_DIR, CHECKPOINT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MONTGOMERY_BASE = "https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Montgomery-County-CXR-Set/MontgomerySet"
SHENZHEN_BASE = "https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Shenzhen-Hospital-CXR-Set"

print(f"Device: {DEVICE}")
print(f"Project dir: {PROJECT_DIR}")
print(f"Data dir: {DATA_DIR}")

Device: cuda
Project dir: e:\Github\Machine-Learning-Projects\Computer Vision\Lung Segmentation from Chest X-Ray\Source Code
Data dir: e:\Github\Machine-Learning-Projects\Computer Vision\Lung Segmentation from Chest X-Ray\Source Code\data


## Real Dataset Download

This notebook downloads data from the NLM public directory listings instead of assuming files already exist.

Download plan:
- **Montgomery:** all images plus left and right lung masks
- **Shenzhen:** a small real subset of images for external qualitative inference

Why not train on Shenzhen here:
- The official NLM Shenzhen release used here exposes image files and abnormality annotation masks, but not paired lung-field masks in the same straightforward form as Montgomery.
- A lung segmentation notebook should not pretend those are interchangeable labels.

The code below scrapes the public file listings, downloads the PNG files, and verifies counts before moving on.

In [4]:
def list_png_filenames(index_url: str) -> list[str]:
    response = requests.get(index_url, timeout=60)
    response.raise_for_status()
    names = re.findall(r'(?:MCUCXR|CHNCXR)_[0-9]+_[01]\.png', response.text)
    return sorted(set(names))


def download_file(url: str, destination: Path) -> None:
    if destination.exists():
        return
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    destination.write_bytes(response.content)


montgomery_image_names = list_png_filenames(f"{MONTGOMERY_BASE}/CXR_png/index.html")
montgomery_left_names = list_png_filenames(f"{MONTGOMERY_BASE}/ManualMask/leftMask/index.html")
montgomery_right_names = list_png_filenames(f"{MONTGOMERY_BASE}/ManualMask/rightMask/index.html")
common_montgomery_names = sorted(set(montgomery_image_names) & set(montgomery_left_names) & set(montgomery_right_names))

shenzhen_image_names = list_png_filenames(f"{SHENZHEN_BASE}/CXR_png/index.html")
selected_shenzhen_names = shenzhen_image_names[:SHENZHEN_EXTERNAL_LIMIT]

print(f"Montgomery paired studies: {len(common_montgomery_names)}")
print(f"Shenzhen external images selected: {len(selected_shenzhen_names)}")
print(common_montgomery_names[:3])
print(selected_shenzhen_names[:3])

Montgomery paired studies: 138
Shenzhen external images selected: 8
['MCUCXR_0001_0.png', 'MCUCXR_0002_0.png', 'MCUCXR_0003_0.png']
['CHNCXR_0001_0.png', 'CHNCXR_0002_0.png', 'CHNCXR_0003_0.png']


## Dataset Verification and Exploratory Data Audit

Before training, we verify that:
- every Montgomery image has both a left and right lung mask
- masks and images align by filename
- downloaded files are readable
- intensity ranges and mask coverage look plausible

This is especially important in medical imaging because silent data mismatches produce deceptively strong-looking but invalid results.

In [5]:
montgomery_dir = DATA_DIR / "montgomery"
montgomery_images_dir = montgomery_dir / "images"
montgomery_left_dir = montgomery_dir / "left_masks"
montgomery_right_dir = montgomery_dir / "right_masks"
shenzhen_dir = DATA_DIR / "shenzhen" / "images"
for directory in (montgomery_images_dir, montgomery_left_dir, montgomery_right_dir, shenzhen_dir):
    directory.mkdir(parents=True, exist_ok=True)

for name in common_montgomery_names:
    download_file(f"{MONTGOMERY_BASE}/CXR_png/{name}", montgomery_images_dir / name)
    download_file(f"{MONTGOMERY_BASE}/ManualMask/leftMask/{name}", montgomery_left_dir / name)
    download_file(f"{MONTGOMERY_BASE}/ManualMask/rightMask/{name}", montgomery_right_dir / name)

for name in selected_shenzhen_names:
    download_file(f"{SHENZHEN_BASE}/CXR_png/{name}", shenzhen_dir / name)

records: list[dict] = []
for name in common_montgomery_names:
    records.append(
        {
            "image_name": name,
            "image_path": str(montgomery_images_dir / name),
            "left_mask_path": str(montgomery_left_dir / name),
            "right_mask_path": str(montgomery_right_dir / name),
        }
    )

manifest_df = pd.DataFrame(records)
manifest_df.to_csv(ARTIFACT_DIR / "montgomery_manifest.csv", index=False)

assert len(manifest_df) == len(common_montgomery_names) > 0
assert all(Path(path).exists() for path in manifest_df["image_path"])
assert all(Path(path).exists() for path in manifest_df["left_mask_path"])
assert all(Path(path).exists() for path in manifest_df["right_mask_path"])
assert len(list(shenzhen_dir.glob("*.png"))) == len(selected_shenzhen_names)

sample_row = manifest_df.iloc[0]
sample_image = np.array(Image.open(sample_row["image_path"]).convert("L"))
sample_left = np.array(Image.open(sample_row["left_mask_path"]).convert("L"))
sample_right = np.array(Image.open(sample_row["right_mask_path"]).convert("L"))
sample_mask = np.maximum(sample_left, sample_right)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(sample_image, cmap="gray")
axes[0].set_title("Montgomery image")
axes[1].imshow(sample_mask, cmap="gray")
axes[1].set_title("Combined lung mask")
axes[2].imshow(sample_image, cmap="gray")
axes[2].imshow(np.ma.masked_where(sample_mask == 0, sample_mask), cmap="autumn", alpha=0.45)
axes[2].set_title("Overlay")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "montgomery_data_audit.png", dpi=140, bbox_inches="tight")
plt.close()

print(f"Downloaded Montgomery studies: {len(manifest_df)}")
print(f"Downloaded Shenzhen studies: {len(selected_shenzhen_names)}")
print("Saved montgomery_manifest.csv and montgomery_data_audit.png")

Downloaded Montgomery studies: 138
Downloaded Shenzhen studies: 8
Saved montgomery_manifest.csv and montgomery_data_audit.png


## Train, Validation, and Test Strategy

Montgomery is small, so the split needs to preserve enough data for learning while still reserving honest holdout evaluation.

Split used here:
- 70% train
- 15% validation
- 15% test

Why this is reasonable:
- the dataset is small enough that wasting too much data on validation hurts learning
- a separate test split is still important because medical segmentation can overfit quickly on repeated anatomy patterns

In [6]:
train_df, temp_df = train_test_split(manifest_df, test_size=0.30, random_state=SEED, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, shuffle=True)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

split_summary = {
    "train": int(len(train_df)),
    "val": int(len(val_df)),
    "test": int(len(test_df)),
    "shenzhen_external": int(len(selected_shenzhen_names)),
}
print(split_summary)

with open(ARTIFACT_DIR / "split_summary.json", "w", encoding="utf-8") as handle:
    json.dump(split_summary, handle, indent=2)

{'train': 96, 'val': 21, 'test': 21, 'shenzhen_external': 8}


## MONAI Transform Pipeline

Transform choices:
- `LoadImaged` keeps data handling explicit and medical-image-friendly.
- `EnsureChannelFirstd` converts grayscale arrays into `[C, H, W]` tensors.
- `ScaleIntensityRangePercentilesd` normalizes scanner exposure without relying on fixed absolute intensity assumptions.
- `Resized` standardizes input size for efficient batch training.
- `RandAffined` applies only mild translation and scale perturbation to avoid unrealistic anatomy.
- `Lambdad` binarizes masks after loading so the model sees clean foreground targets.

In [8]:
combined_mask_dir = DATA_DIR / "montgomery_combined_masks"
combined_mask_dir.mkdir(parents=True, exist_ok=True)


def combine_lung_masks(row: pd.Series) -> np.ndarray:
    left_mask = np.array(Image.open(row["left_mask_path"]).convert("L"), dtype=np.uint8)
    right_mask = np.array(Image.open(row["right_mask_path"]).convert("L"), dtype=np.uint8)
    return np.maximum(left_mask, right_mask)


def build_records(frame: pd.DataFrame) -> list[dict]:
    records: list[dict] = []
    for _, row in frame.iterrows():
        combined_mask_path = combined_mask_dir / row["image_name"]
        if not combined_mask_path.exists():
            Image.fromarray(combine_lung_masks(row)).save(combined_mask_path)
        records.append({
            "image": row["image_path"],
            "mask": str(combined_mask_path),
            "image_name": row["image_name"],
        })
    return records


train_records = build_records(train_df)
val_records = build_records(val_df)
test_records = build_records(test_df)

train_transforms = Compose(
    [
        LoadImaged(keys=["image", "mask"], image_only=True),
        EnsureChannelFirstd(keys=["image", "mask"], channel_dim="no_channel"),
        ScaleIntensityRangePercentilesd(
            keys=["image"],
            lower=1,
            upper=99,
            b_min=0.0,
            b_max=1.0,
            clip=True,
        ),
        Resized(keys=["image", "mask"], spatial_size=(IMAGE_SIZE, IMAGE_SIZE), mode=["bilinear", "nearest"]),
        RandAffined(
            keys=["image", "mask"],
            prob=0.30,
            translate_range=(8, 8),
            scale_range=(0.04, 0.04),
            mode=["bilinear", "nearest"],
            padding_mode="zeros",
        ),
        Lambdad(keys=["mask"], func=lambda value: (value > 0).astype(np.float32)),
        EnsureTyped(keys=["image", "mask"], dtype=torch.float32),
    ]
)

eval_transforms = Compose(
    [
        LoadImaged(keys=["image", "mask"], image_only=True),
        EnsureChannelFirstd(keys=["image", "mask"], channel_dim="no_channel"),
        ScaleIntensityRangePercentilesd(
            keys=["image"],
            lower=1,
            upper=99,
            b_min=0.0,
            b_max=1.0,
            clip=True,
        ),
        Resized(keys=["image", "mask"], spatial_size=(IMAGE_SIZE, IMAGE_SIZE), mode=["bilinear", "nearest"]),
        Lambdad(keys=["mask"], func=lambda value: (value > 0).astype(np.float32)),
        EnsureTyped(keys=["image", "mask"], dtype=torch.float32),
    ]
)

train_dataset = CacheDataset(data=train_records, transform=train_transforms, cache_rate=1.0, num_workers=0)
val_dataset = CacheDataset(data=val_records, transform=eval_transforms, cache_rate=1.0, num_workers=0)
test_dataset = CacheDataset(data=test_records, transform=eval_transforms, cache_rate=1.0, num_workers=0)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print({
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
    "test_samples": len(test_dataset),
})

Loading dataset: 100%|██████████| 21/21 [00:07<00:00,  2.67it/s]

{'train_samples': 96, 'val_samples': 21, 'test_samples': 21}


## MONAI UNet, Loss, and Metric Setup

This notebook uses a lightweight 2D MONAI UNet because the Montgomery set is small and grayscale. `DiceCELoss` stabilizes optimization by combining overlap quality with pixel-wise supervision, while validation tracks both Dice and IoU so the notebook reports the standard medical-segmentation overlap metrics explicitly.

In [9]:
def batch_iou(predictions: torch.Tensor, targets: torch.Tensor, eps: float = 1e-6) -> float:
    intersection = (predictions * targets).sum(dim=(1, 2, 3))
    union = ((predictions + targets) > 0).float().sum(dim=(1, 2, 3))
    return float(((intersection + eps) / (union + eps)).mean().item())


model = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(DEVICE)

loss_function = DiceCELoss(sigmoid=True)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
dice_metric = DiceMetric(include_background=True, reduction="mean")

print(model.__class__.__name__)
print(f"Trainable parameters: {sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad):,}")

UNet
Trainable parameters: 1,624,844


## Training Loop and Learning Curves

The training run is intentionally short so the notebook remains practical to execute locally. Even with a short run, the notebook still saves the best checkpoint by validation Dice and exports a curve plot so you can inspect whether overlap quality is improving or flattening.

In [10]:
history: list[dict] = []
best_val_dice = -1.0
best_checkpoint_path = CHECKPOINT_DIR / "best_monai_unet_lung_segmentation.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_train_loss = 0.0
    train_batches = 0
    for batch in train_loader:
        images = batch["image"].to(DEVICE)
        masks = batch["mask"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = loss_function(logits, masks)
        loss.backward()
        optimizer.step()

        running_train_loss += float(loss.item())
        train_batches += 1

    model.eval()
    running_val_loss = 0.0
    running_val_iou = 0.0
    val_batches = 0
    dice_metric.reset()
    with torch.no_grad():
        for batch in val_loader:
            images = batch["image"].to(DEVICE)
            masks = batch["mask"].to(DEVICE)
            logits = model(images)
            loss = loss_function(logits, masks)
            predictions = (torch.sigmoid(logits) > 0.5).float()

            running_val_loss += float(loss.item())
            running_val_iou += batch_iou(predictions, masks)
            dice_metric(y_pred=predictions, y=masks)
            val_batches += 1

    mean_train_loss = running_train_loss / max(train_batches, 1)
    mean_val_loss = running_val_loss / max(val_batches, 1)
    mean_val_iou = running_val_iou / max(val_batches, 1)
    mean_val_dice = float(dice_metric.aggregate().item())
    dice_metric.reset()

    epoch_record = {
        "epoch": epoch,
        "train_loss": mean_train_loss,
        "val_loss": mean_val_loss,
        "val_dice": mean_val_dice,
        "val_iou": mean_val_iou,
    }
    history.append(epoch_record)
    print(epoch_record)

    if mean_val_dice > best_val_dice:
        best_val_dice = mean_val_dice
        torch.save(model.state_dict(), best_checkpoint_path)

history_df = pd.DataFrame(history)
history_df.to_csv(ARTIFACT_DIR / "training_history.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train loss")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Val loss")
axes[0].set_title("Loss curves")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(history_df["epoch"], history_df["val_dice"], marker="o", label="Val Dice")
axes[1].plot(history_df["epoch"], history_df["val_iou"], marker="o", label="Val IoU")
axes[1].set_title("Validation overlap metrics")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "training_curves.png", dpi=140, bbox_inches="tight")
plt.close()

print(f"Best checkpoint: {best_checkpoint_path}")
history_df

{'epoch': 1, 'train_loss': 1.0438608552018802, 'val_loss': 0.8982687890529633, 'val_dice': 0.8495122194290161, 'val_iou': 0.7680612504482269}
{'epoch': 2, 'train_loss': 0.8441475729147593, 'val_loss': 0.8187141418457031, 'val_dice': 0.8774012327194214, 'val_iou': 0.8096647063891093}
Best checkpoint: e:\Github\Machine-Learning-Projects\Computer Vision\Lung Segmentation from Chest X-Ray\Source Code\checkpoint\best_monai_unet_lung_segmentation.pth


,epoch,train_loss,val_loss,val_dice,val_iou
0,1,1.043861,0.898269,0.849512,0.768061
1,2,0.844148,0.818714,0.877401,0.809665


## Test Evaluation and Qualitative Review

The Montgomery test split provides the supervised Dice and IoU scores. Shenzhen is kept separate as an external qualitative stress check so the notebook stays honest about where ground-truth masks exist and where they do not.

In [11]:
model.load_state_dict(torch.load(best_checkpoint_path, map_location=DEVICE))
model.eval()

dice_metric.reset()
test_loss = 0.0
test_iou = 0.0
test_batches = 0
with torch.no_grad():
    for batch in test_loader:
        images = batch["image"].to(DEVICE)
        masks = batch["mask"].to(DEVICE)
        logits = model(images)
        predictions = (torch.sigmoid(logits) > 0.5).float()
        test_loss += float(loss_function(logits, masks).item())
        test_iou += batch_iou(predictions, masks)
        dice_metric(y_pred=predictions, y=masks)
        test_batches += 1

test_metrics = {
    "test_loss": test_loss / max(test_batches, 1),
    "test_dice": float(dice_metric.aggregate().item()),
    "test_iou": test_iou / max(test_batches, 1),
    "best_val_dice": best_val_dice,
    "epochs": EPOCHS,
    "image_size": IMAGE_SIZE,
}
dice_metric.reset()

fig, axes = plt.subplots(min(3, len(test_dataset)), 3, figsize=(10, 9))
if len(test_dataset) == 1:
    axes = np.expand_dims(axes, axis=0)
for index in range(min(3, len(test_dataset))):
    sample = test_dataset[index]
    image_tensor = sample["image"].unsqueeze(0).to(DEVICE)
    mask_array = sample["mask"].squeeze().cpu().numpy()
    with torch.no_grad():
        prediction_array = (torch.sigmoid(model(image_tensor)).squeeze().cpu().numpy() > 0.5).astype(np.float32)
    image_array = sample["image"].squeeze().cpu().numpy()
    axes[index, 0].imshow(image_array, cmap="gray")
    axes[index, 0].set_title(f"Image {index + 1}")
    axes[index, 1].imshow(mask_array, cmap="gray")
    axes[index, 1].set_title("Ground truth")
    axes[index, 2].imshow(prediction_array, cmap="gray")
    axes[index, 2].set_title("Prediction")
    for axis in axes[index]:
        axis.axis("off")
plt.tight_layout()
montgomery_review_path = ARTIFACT_DIR / "montgomery_test_predictions.png"
plt.savefig(montgomery_review_path, dpi=140, bbox_inches="tight")
plt.close()

external_transform = Compose(
    [
        LoadImaged(keys=["image"], image_only=True),
        EnsureChannelFirstd(keys=["image"], channel_dim="no_channel"),
        ScaleIntensityRangePercentilesd(
            keys=["image"],
            lower=1,
            upper=99,
            b_min=0.0,
            b_max=1.0,
            clip=True,
        ),
        Resized(keys=["image"], spatial_size=(IMAGE_SIZE, IMAGE_SIZE), mode=["bilinear"]),
        EnsureTyped(keys=["image"], dtype=torch.float32),
    ]
)

fig, axes = plt.subplots(min(3, len(selected_shenzhen_names)), 2, figsize=(8, 9))
if len(selected_shenzhen_names) == 1:
    axes = np.expand_dims(axes, axis=0)
for index in range(min(3, len(selected_shenzhen_names))):
    image_path = shenzhen_dir / selected_shenzhen_names[index]
    transformed = external_transform({"image": str(image_path)})
    image_tensor = transformed["image"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prediction_array = (torch.sigmoid(model(image_tensor)).squeeze().cpu().numpy() > 0.5).astype(np.float32)
    image_array = transformed["image"].squeeze().cpu().numpy()
    axes[index, 0].imshow(image_array, cmap="gray")
    axes[index, 0].set_title(f"Shenzhen {index + 1}")
    axes[index, 1].imshow(image_array, cmap="gray")
    axes[index, 1].imshow(np.ma.masked_where(prediction_array == 0, prediction_array), cmap="autumn", alpha=0.45)
    axes[index, 1].set_title("Predicted lung field")
    for axis in axes[index]:
        axis.axis("off")
plt.tight_layout()
shenzhen_review_path = ARTIFACT_DIR / "shenzhen_external_predictions.png"
plt.savefig(shenzhen_review_path, dpi=140, bbox_inches="tight")
plt.close()

metrics = {
    "project": "Lung Segmentation from Chest X-Ray",
    "task": "lung_field_segmentation",
    "supervised_dataset": "Montgomery County CXR Set",
    "external_qualitative_dataset": "Shenzhen Hospital CXR Set",
    "model": "MONAI UNet",
    **test_metrics,
}
metrics_path = ARTIFACT_DIR / "metrics.json"
with open(metrics_path, "w", encoding="utf-8") as handle:
    json.dump(metrics, handle, indent=2)

print(metrics)
print(f"Saved {montgomery_review_path.name}, {shenzhen_review_path.name}, and metrics.json")

{'project': 'Lung Segmentation from Chest X-Ray', 'task': 'lung_field_segmentation', 'supervised_dataset': 'Montgomery County CXR Set', 'external_qualitative_dataset': 'Shenzhen Hospital CXR Set', 'model': 'MONAI UNet', 'test_loss': 0.7603432138760885, 'test_dice': 0.9268065094947815, 'test_iou': 0.8726416528224945, 'best_val_dice': 0.8774012327194214, 'epochs': 2, 'image_size': 256}
Saved montgomery_test_predictions.png, shenzhen_external_predictions.png, and metrics.json


## Artifact Manifest and Medical-Imaging Notes

The saved manifest records what was trained, what was only used for external qualitative review, and why the preprocessing choices are conservative. For chest radiographs, preserving anatomy is more important than aggressive augmentation, so this notebook intentionally stays close to clinically plausible image geometry.

In [12]:
artifact_manifest = {
    "project": "Lung Segmentation from Chest X-Ray",
    "problem_type": "medical image segmentation",
    "framework": "PyTorch + MONAI",
    "supervised_dataset": {
        "name": "Montgomery County CXR Set",
        "source": MONTGOMERY_BASE,
        "usage": "training, validation, and test Dice/IoU evaluation",
    },
    "external_dataset": {
        "name": "Shenzhen Hospital CXR Set",
        "source": SHENZHEN_BASE,
        "usage": "qualitative external inference only",
    },
    "preprocessing_notes": [
        "Single-channel grayscale pipeline.",
        "Percentile-based intensity scaling to reduce exposure variability.",
        "Nearest-neighbor mask resizing to preserve binary labels.",
        "Only mild affine augmentation to avoid anatomy-breaking distortions.",
    ],
    "artifacts": {
        "manifest_csv": str(ARTIFACT_DIR / "montgomery_manifest.csv"),
        "split_summary": str(ARTIFACT_DIR / "split_summary.json"),
        "training_history": str(ARTIFACT_DIR / "training_history.csv"),
        "training_curves": str(ARTIFACT_DIR / "training_curves.png"),
        "montgomery_review": str(montgomery_review_path),
        "shenzhen_review": str(shenzhen_review_path),
        "metrics": str(metrics_path),
        "best_checkpoint": str(best_checkpoint_path),
    },
}

manifest_path = ARTIFACT_DIR / "artifact_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as handle:
    json.dump(artifact_manifest, handle, indent=2)

print(f"Saved manifest to {manifest_path}")
print(f"Checkpoint stored at {best_checkpoint_path}")

Saved manifest to e:\Github\Machine-Learning-Projects\Computer Vision\Lung Segmentation from Chest X-Ray\Source Code\artifacts\artifact_manifest.json
Checkpoint stored at e:\Github\Machine-Learning-Projects\Computer Vision\Lung Segmentation from Chest X-Ray\Source Code\checkpoint\best_monai_unet_lung_segmentation.pth


## Limitations, Improvements, and Takeaways

Limitations:
- Montgomery is a small dataset, so this notebook is best treated as a compact medical-segmentation lab rather than a production-ready benchmark.
- Shenzhen is used only for qualitative transfer checks because the public release does not provide matching lung masks in the same format as Montgomery.
- Two training epochs are enough for validation of the workflow, not for best possible overlap performance.

How to improve this project:
- Train longer with early stopping and a richer validation schedule.
- Compare MONAI UNet against attention UNet or a stronger encoder-decoder baseline.
- Add patient-level external validation on another public chest-radiograph segmentation set.
- Track calibration and boundary quality metrics in addition to Dice and IoU.

Key takeaway:
- MONAI gives a clean path for medical-image preprocessing, segmentation training, and honest evaluation when the notebook clearly separates supervised overlap scoring from external qualitative review.